# 01 — Load and clean source data

**Purpose:** load eight semicolon-delimited Berka files into the MySQL raw layer.
This notebook replaces exact missing-value sentinels, maps column names, selects
configured fields, and casts types before writing in foreign-key dependency order.

**Prerequisites:** follow [the setup guide](../docs/SETUP.md), select the project
kernel with `notebooks/` as its working directory, and create empty raw tables.
The loader expects `district.csv`; rename upstream `district.asc` accordingly.

**Writes:** appends to eight database tables. Run once against empty tables;
repeating or resuming a partial run can fail on primary-key conflicts. The loop's
completion messages do not replace the row counts and quality checks in `03`.
No saved execution outputs are included in this notebook.


In [ ]:
import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_raw_csv, prepare_table, write_to_mysql
from src.table_configs import TABLES, LOAD_ORDER, rename_map, type_map

In [ ]:
cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

In [ ]:
for table_key in LOAD_ORDER:
    table_cfg = TABLES[table_key]
    file_path = cfg["raw_dir"] / table_cfg["file"]

    print(f"Loading: {table_key} <- {table_cfg['file']}")

    raw_df = read_raw_csv(spark, file_path)

    clean_df = prepare_table(
        raw_df,
        col_rename=rename_map(table_key),
        col_types=type_map(table_key),
    )

    write_to_mysql(
        clean_df,
        table_cfg["db_table"],
        cfg["jdbc_url"],
        cfg["db_user"],
        cfg["db_password"],
    )

    print(f"Done: {table_key}")

In [ ]:
spark.stop()